# SLC46A1 engraftment, step 9: the revised construct scored as a set

Step 8 found a rescue for chicken 407 in the background the construct has, A404D with
V406R at 1.070241e-03, where the pair chosen in step 4 against wild-type chicken reached
only 6.771676e-04 there. Taking it swaps chicken 410 back to its chicken residue and
writes a rescuer at chicken 406, so the construct now differs at two positions from the
one step 7 scored, and step 7's set-level result does not carry over.

This run scores the revised construct the way step 7 scored the old one. Every one of the
20 humanising substitutions is read with the other nineteen and all six rescuers in place,
giving the profile, and the same positions are read in two reference backgrounds, wild-type
chicken and the primaries alone, which reproduce step 1 and step 5 and put this run on
their scale. Sixty forward passes in all.

One check is exact rather than approximate. The revised construct with chicken 407 left
carrying its chicken residue is step 8's background with A404D and V406R written in, so
rebuilding it here has to give step 8's background byte for byte, and scoring it has to
reproduce step 8's 1.070241e-03. Both run before the sweep.

Every result is written to Drive as it is produced and the run resumes from its
checkpoint.

In [ ]:
import os, sys, json, hashlib, datetime, time

from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/MIT/All Project Files/SLC46A1 engraftment'
OUT     = os.path.join(PROJECT, 'data', 'esm2_step9')

if not os.path.isdir(PROJECT):
    raise SystemExit('FATAL project folder not found: %s' % PROJECT)
os.makedirs(OUT, exist_ok=True)

REQUIRED = ['data/admissible_construct.json',
            'data/construct_report.json',
            'data/pharmacophore_position_set.tsv',
            'data/primary_substitution_risk.tsv']
missing = [f for f in REQUIRED if not os.path.exists(os.path.join(PROJECT, f))]
if missing:
    raise SystemExit('FATAL missing inputs in %s: %s' % (PROJECT, missing))


def resolve(name, candidates):
    found = [f for f in candidates if os.path.exists(os.path.join(PROJECT, f))]
    if not found:
        raise SystemExit('FATAL %s not found in %s under any of %s. This run is tied to '
                         'it and cannot start without it.' % (name, PROJECT, candidates))
    if len(found) > 1:
        digests = {f: hashlib.sha256(open(os.path.join(PROJECT, f), 'rb').read())
                   .hexdigest() for f in found}
        if len(set(digests.values())) > 1:
            raise SystemExit('FATAL two different copies of %s are on Drive, %s. Delete '
                             'the stale one so there is one copy to tie this run to.'
                             % (name, digests))
    return os.path.join(PROJECT, found[0]), found[0]


S7_PATH, s7rel = resolve('the step 7 summary',
                         ['data/esm2_step7/step7_summary.json',
                          'esm2_step7/step7_summary.json'])
S8_PATH, s8rel = resolve('the step 8 summary',
                         ['data/esm2_step8/step8_summary.json',
                          'esm2_step8/step8_summary.json'])

RUNLOG = os.path.join(OUT, 'run.log')


def log(msg):
    line = '%s  %s' % (datetime.datetime.now(datetime.timezone.utc)
                       .isoformat(timespec='seconds'), msg)
    with open(RUNLOG, 'a') as fh:
        fh.write(line + chr(10))
    print(line, flush=True)


log('start, project=%s' % PROJECT)
log('step 7 summary from %s' % s7rel)
log('step 8 summary from %s' % s8rel)
log('output folder %s' % OUT)


In [ ]:
import shutil, subprocess

if shutil.which('nvidia-smi') is None:
    raise SystemExit('FATAL this runtime has no GPU. Set Runtime > Change runtime '
                     'type > T4 GPU (or better), then run from the first cell again.')
if subprocess.run(['nvidia-smi'], capture_output=True).returncode != 0:
    raise SystemExit('FATAL nvidia-smi is present but failing, so the runtime has no '
                     'usable GPU. Change the runtime type and run from the first cell.')

subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'torch', 'transformers'],
               check=True)

import torch
from transformers import AutoTokenizer, EsmForMaskedLM

if not torch.cuda.is_available():
    raise SystemExit('FATAL torch cannot see the GPU after install')
log('torch %s, device %s' % (torch.__version__, torch.cuda.get_device_name(0)))


In [ ]:
import urllib.request

SEQ = os.path.join(OUT, 'seq')
os.makedirs(SEQ, exist_ok=True)


def fetch(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    urllib.request.urlretrieve(url, path)
    if os.path.getsize(path) == 0:
        raise SystemExit('FATAL empty download: %s' % url)
    return path


fetch('https://rest.uniprot.org/uniprotkb/Q96NT5.fasta', os.path.join(SEQ, 'Q96NT5.fasta'))
fetch('https://www.rcsb.org/fasta/entry/7BC6', os.path.join(SEQ, '7BC6.fasta'))

human = ''.join(l.strip() for l in open(os.path.join(SEQ, 'Q96NT5.fasta'))
                if not l.startswith('>'))
ent = {}
for rec in open(os.path.join(SEQ, '7BC6.fasta')).read().split('>'):
    if rec.strip():
        parts = rec.strip().split(chr(10))
        ent[parts[0].split('|')[0]] = ''.join(parts[1:])
chicken = ent['7BC6_1'][:473]
if len(human) != 459 or len(chicken) != 473:
    raise SystemExit('FATAL sequence lengths %d and %d, expected 459 and 473'
                     % (len(human), len(chicken)))
CHICKEN_SHA = hashlib.sha256(chicken.encode()).hexdigest()
EXPECT_SHA = '5d68e7c91dc3af3cd898a3f459617f3b22670f3dbdf9f798e8d740d8669a8aea'
if CHICKEN_SHA != EXPECT_SHA:
    raise SystemExit('FATAL the chicken sequence differs from the one the earlier steps '
                     'scored: %s against %s' % (CHICKEN_SHA, EXPECT_SHA))

AA = 'ACDEFGHIKLMNPQRSTVWY'
THRESHOLD = 1e-3
TOL = 5e-2

rows = [l.rstrip(chr(10)).split(chr(9))
        for l in open(os.path.join(PROJECT, 'data', 'pharmacophore_position_set.tsv'))]
pcol = {k: i for i, k in enumerate(rows[0])}
PSET = {}
for r in rows[1:]:
    cp, hp = int(r[pcol['chicken_pos']]), int(r[pcol['human_pos']])
    if chicken[cp - 1] != r[pcol['chicken_aa']] or human[hp - 1] != r[pcol['human_aa']]:
        raise SystemExit('FATAL the position set disagrees with the sequences at chicken '
                         '%d' % cp)
    PSET[cp] = {'chicken_aa': r[pcol['chicken_aa']], 'human_aa': r[pcol['human_aa']],
                'human_pos': hp, 'differs': r[pcol['differs']] == '1'}
PRIM = sorted(p for p, v in PSET.items() if v['differs'])
if len(PRIM) != 20:
    raise SystemExit('FATAL %d primaries, expected 20' % len(PRIM))

adm = json.load(open(os.path.join(PROJECT, 'data', 'admissible_construct.json')))
if sorted(adm['primaries']) != PRIM:
    raise SystemExit('FATAL the admissible construct lists different primaries from the '
                     'position set')
RESC = {int(k): v for k, v in adm['rescuer_positions'].items()}
for q, y in RESC.items():
    if q in PSET:
        raise SystemExit('FATAL rescuer position %d is in the pharmacophore set, so the '
                         'construct is not the admissible one' % q)
    if y == chicken[q - 1]:
        raise SystemExit('FATAL the rescuer at %d is not a substitution' % q)

s7 = json.load(open(S7_PATH))
s8 = json.load(open(S8_PATH))
for name, blob in (('step 7', s7), ('step 8', s8)):
    if blob['chicken_sha256'] != CHICKEN_SHA:
        raise SystemExit('FATAL %s scored a different chicken sequence' % name)
S7P = {x['chicken_pos']: x for x in s7['part_a']['profile']}
TARGET8 = int(s8['target'])
BEST8 = s8['best_sparing_double']
if BEST8 is None:
    raise SystemExit('FATAL step 8 recorded no admissible crossing pair, so the construct '
                     'this run scores cannot have come from it')

con = json.load(open(os.path.join(PROJECT, 'data', 'construct_report.json')))
PROF5 = {x['chicken_pos']: x for x in con['part_b']['profile']}
rows = [l.rstrip(chr(10)).split(chr(9))
        for l in open(os.path.join(PROJECT, 'data',
                                   'primary_substitution_risk.tsv'))]
rcol = {k: i for i, k in enumerate(rows[0])}
WT_BASE = {int(r[rcol['chicken_pos']]): float(r[rcol['p_human_wt_background']])
           for r in rows[1:]}
missing = [t for t in PRIM if t not in WT_BASE or t not in PROF5 or t not in S7P]
if missing:
    raise SystemExit('FATAL no earlier value on record for chicken %s, so this run cannot '
                     'be put on the earlier scale' % missing)
DISPLACED = adm.get('overruled_by_background', [])
SWAPPED = sorted({q for d in DISPLACED for q in d['positions_swapped']})
if not DISPLACED:
    raise SystemExit('FATAL the construct file records no rescue displaced by a '
                     'construct-background measurement, so it is the construct step 7 '
                     'already scored and there is nothing for this run to re-score.')
log('revised construct: %d primaries, %d rescuer positions %s'
    % (len(PRIM), len(RESC), sorted(RESC)))
for d in DISPLACED:
    log('  chicken %d now carries %s at %.6e in %s, in place of %s at %.6e in %s'
        % (d['target'], d['chosen'], d['chosen_p'], d['chosen_background'],
           d['strongest'], d['strongest_p'], d['strongest_background']))
log('  rescuer positions that changed since step 7: %s' % SWAPPED)


In [ ]:
MODEL = 'facebook/esm2_t33_650M_UR50D'
BATCH = 48
SCHEMA = 'v9-revised-construct-as-a-set'

tok = AutoTokenizer.from_pretrained(MODEL)
model = EsmForMaskedLM.from_pretrained(MODEL).eval().cuda()
AAID = {y: tok.convert_tokens_to_ids(y) for y in AA}
if any(v is None or v < 0 for v in AAID.values()):
    raise SystemExit('FATAL the tokenizer has no id for one of the amino acids')
offset = 1
NTOK = len(chicken) + 2


def build(primaries_in, rescuers_in, leave_out=None, drop=()):
    s = list(chicken)
    if primaries_in:
        for p in PRIM:
            if p != leave_out:
                s[p - 1] = PSET[p]['human_aa']
    if rescuers_in:
        for q, y in RESC.items():
            if q != leave_out and q not in drop:
                s[q - 1] = y
    return ''.join(s)


REVISED = build(True, True)
if sum(1 for a, b in zip(REVISED, chicken) if a != b) != adm['n_positions_changed']:
    raise SystemExit('FATAL the rebuilt construct does not differ from chicken at the '
                     '%d positions the construct file records'
                     % adm['n_positions_changed'])
if REVISED != adm['sequence']:
    raise SystemExit('FATAL the rebuilt construct is not the stored one')

CH8 = adm['chosen'][str(TARGET8)]
R8 = [int(p) for p in CH8['positions']]
BG8 = build(True, True, leave_out=TARGET8, drop=set(R8))
if BG8 != s8['background']['sequence']:
    n = sum(1 for a, b in zip(BG8, s8['background']['sequence']) if a != b)
    raise SystemExit('FATAL this construct minus the rescue at chicken %d is not the '
                     'background step 8 searched, %d positions differ. The two runs are '
                     'not about the same construct.' % (TARGET8, n))
log('check           : this construct minus %s rebuilds step 8 background byte for byte'
    % CH8['label'])


def score(jobs):
    out = []
    for b in range(0, len(jobs), BATCH):
        chunk = jobs[b:b + BATCH]
        enc = tok([s for s, _ in chunk], return_tensors='pt')['input_ids']
        if enc.shape[1] != NTOK:
            raise SystemExit('FATAL a background tokenizes to %d tokens, expected %d'
                             % (enc.shape[1], NTOK))
        with torch.no_grad():
            logits = model(enc.cuda()).logits
        for k, (_, t) in enumerate(chunk):
            p = torch.softmax(logits[k, t - 1 + offset, :].float(), dim=-1)
            out.append(float(p[AAID[PSET[t]['human_aa']]]))
    return out


CHECK8 = score([(build(True, True, leave_out=TARGET8), TARGET8)])[0]
rel = abs(CHECK8 - BEST8['p_target']) / max(CHECK8, BEST8['p_target'], 1e-30)
log('chicken %d in this construct    : %.6e' % (TARGET8, CHECK8))
log('the same variant in step 8      : %.6e, relative difference %.2e'
    % (BEST8['p_target'], rel))
if rel > TOL:
    raise SystemExit('FATAL this run does not reproduce step 8 at chicken %d, %.6e here '
                     'against %.6e there, so the construct it scores is not the one step '
                     '8 chose.' % (TARGET8, CHECK8, BEST8['p_target']))
log('check           : the run reproduces step 8 at chicken %d' % TARGET8)


In [ ]:
BACKGROUNDS = [('wt', False, False), ('primaries', True, False),
               ('revised', True, True)]
jobs = [(tag, t) for tag, pr, rs in BACKGROUNDS for t in PRIM]
log('backgrounds %d, primaries %d, forward passes %d'
    % (len(BACKGROUNDS), len(PRIM), len(jobs)))

CK = os.path.join(OUT, 'profile_partial.tsv')
HEAD = chr(9).join(['schema_' + SCHEMA, 'background', 'chicken_pos', 'p_target'])
done = {}
if os.path.exists(CK) and os.path.getsize(CK) > 0:
    if open(CK).readline().rstrip(chr(10)) != HEAD:
        raise SystemExit('FATAL checkpoint %s was written by a different schema; delete '
                         'it and rerun' % CK)
    for line in open(CK):
        p = line.rstrip(chr(10)).split(chr(9))
        if p[0] == 'row':
            done[(p[1], int(p[2]))] = float(p[3])
    log('resuming, already scored %d of %d' % (len(done), len(jobs)))

todo = [j for j in jobs if j not in done]
BY_TAG = {tag: (pr, rs) for tag, pr, rs in BACKGROUNDS}
t0 = time.time()
with open(CK, 'a') as fh:
    if not done:
        fh.write(HEAD + chr(10))
        fh.flush()
    for b in range(0, len(todo), BATCH):
        chunk = todo[b:b + BATCH]
        built = [(build(BY_TAG[tag][0], BY_TAG[tag][1], leave_out=t), t)
                 for tag, t in chunk]
        vals = score(built)
        for (tag, t), p in zip(chunk, vals):
            done[(tag, t)] = p
            fh.write(chr(9).join(['row', tag, str(t), '%.6e' % p]) + chr(10))
        fh.flush()
        log('  %d of %d, %.1f min elapsed'
            % (b + len(chunk), len(todo), (time.time() - t0) / 60))
log('sweep complete, %.1f min' % ((time.time() - t0) / 60))
if len(done) != len(jobs):
    raise SystemExit('FATAL the checkpoint holds %d rows, %d expected'
                     % (len(done), len(jobs)))


In [ ]:
worst_wt = (0.0, None)
worst_pr = (0.0, None)
for t in PRIM:
    a, b = done[('wt', t)], WT_BASE[t]
    rel = abs(a - b) / max(a, b, 1e-30)
    if rel > worst_wt[0]:
        worst_wt = (rel, (t, a, b))
    a, b = done[('primaries', t)], PROF5[t]['primaries']
    rel = abs(a - b) / max(a, b, 1e-30)
    if rel > worst_pr[0]:
        worst_pr = (rel, (t, a, b))
if worst_wt[0] > TOL:
    t, a, b = worst_wt[1]
    raise SystemExit('FATAL the wild-type column does not reproduce step 1 at chicken %d, '
                     '%.6e against %.6e' % (t, a, b))
if worst_pr[0] > TOL:
    t, a, b = worst_pr[1]
    raise SystemExit('FATAL the primaries column does not reproduce step 5 at chicken %d, '
                     '%.6e against %.6e' % (t, a, b))
log('scale checks: wild type against step 1 worst %.2e, primaries against step 5 worst '
    '%.2e' % (worst_wt[0], worst_pr[0]))

cols = ['chicken_pos', 'human_aa', 'p_wt', 'p_primaries', 'p_revised', 'p_step7',
        'fold_revised_over_wt', 'fold_revised_over_step7']
prof = []
for t in PRIM:
    prof.append({'chicken_pos': t, 'human_aa': PSET[t]['human_aa'],
                 'p_wt': done[('wt', t)], 'p_primaries': done[('primaries', t)],
                 'p_revised': done[('revised', t)],
                 'p_step7': S7P[t]['p_admissible'],
                 'fold_revised_over_wt': done[('revised', t)] / done[('wt', t)],
                 'fold_revised_over_step7':
                     done[('revised', t)] / S7P[t]['p_admissible']})
with open(os.path.join(OUT, 'revised_profile.tsv'), 'w') as fh:
    fh.write(chr(9).join(cols) + chr(10))
    for r in prof:
        fh.write(chr(9).join(('%.6e' % r[c]) if isinstance(r[c], float) else str(r[c])
                             for c in cols) + chr(10))

log('')
log('every primary with the other nineteen and all %d rescuers in place' % len(RESC))
log('chicken  human    wild type   primaries     revised      step 7   fold over step 7')
for r in prof:
    log('%7d %6s %12.3e %11.3e %11.3e %11.3e %17.2f'
        % (r['chicken_pos'], r['human_aa'], r['p_wt'], r['p_primaries'],
           r['p_revised'], r['p_step7'], r['fold_revised_over_step7']))

below = [r['chicken_pos'] for r in prof if r['p_revised'] < THRESHOLD]
below7 = [r['chicken_pos'] for r in prof if r['p_step7'] < THRESHOLD]
folds = sorted(r['fold_revised_over_step7'] for r in prof)
log('')
log('revised construct below %.0e at %d of %d primaries%s'
    % (THRESHOLD, len(below), len(prof), (': %s' % below) if below else ''))
log('the construct step 7 scored was below it at %d%s'
    % (len(below7), (': %s' % below7) if below7 else ''))
log('fold over step 7 across the primaries: median %.2f, lowest %.2f, highest %.2f'
    % (folds[len(folds) // 2], folds[0], folds[-1]))

surv = []
for t in sorted(int(k) for k in adm['chosen']):
    ch = adm['chosen'][str(t)]
    r = [x for x in prof if x['chicken_pos'] == t][0]
    surv.append({'target': t, 'label': ch['label'], 'chosen_at': ch['p'],
                 'chosen_background': ch['background'],
                 'in_construct': r['p_revised'],
                 'fold': r['p_revised'] / ch['p'],
                 'holds': r['p_revised'] >= THRESHOLD})
log('')
log('does each rescue hold with the others present')
for h in surv:
    log('  chicken %d: %s, chosen at %.6e in %s, here %.6e, %s'
        % (h['target'], h['label'], h['chosen_at'], h['chosen_background'],
           h['in_construct'], 'holds' if h['holds'] else 'does not hold'))
hold = [h['target'] for h in surv if h['holds']]
log('rescues holding: %s of %s' % (hold, [h['target'] for h in surv]))
log('still without an admissible rescue: %s' % (adm['unrescued'] or 'none'))

summary = {'schema': SCHEMA, 'model': MODEL, 'chicken_sha256': CHICKEN_SHA,
           'threshold': THRESHOLD,
           'construct': {'primaries': PRIM,
                         'rescuer_positions': {str(k): v for k, v in RESC.items()},
                         'n_positions_changed': adm['n_positions_changed'],
                         'sequence': REVISED},
           'displaced_rescues': DISPLACED,
           'rescuer_positions_swapped_since_step7': SWAPPED,
           'checks': {'wt_vs_step1_worst': worst_wt[0],
                      'primaries_vs_step5_worst': worst_pr[0],
                      'step8_value': BEST8['p_target'], 'step8_here': CHECK8,
                      'step8_relative_difference': rel,
                      'rebuilds_step8_background': True},
           'profile': prof,
           'below_threshold': below, 'below_threshold_step7': below7,
           'median_fold_over_step7': folds[len(folds) // 2],
           'rescue_survival': surv,
           'rescues_holding': hold,
           'unrescued': adm['unrescued']}
with open(os.path.join(OUT, 'step9_summary.json'), 'w') as fh:
    json.dump(summary, fh, indent=1)
log('')
log('wrote revised_profile.tsv and step9_summary.json')
log('copy data/esm2_step9/ from Drive into the local data/ and run '
    'codes/23_step9_read.py')
